# Segment 4 - Progressive Tool Loading & MCP Clients

**50-minute segment.** The advanced close: how to keep 28 tools from blowing your context budget (progressive loading), and how the same server wires into three different clients (Inspector, Claude Code, GitHub Copilot in VS Code).

**What you'll show:**
1. `warn_search_tools` - the discovery tool that loads schemas on demand (the "code execution with MCP" pattern)
2. The default-limit gotcha: a no-query search returns `count=20` of `total=28`, not 26
3. The two MCP client config schemas side by side (`.claude/mcp.json` vs `.vscode/mcp.json`)

> Anchor cell first. Pick the **Python 3.13** kernel.

## Setup: anchor to the repo root

Same backbone, plus the `warnerco_py` helper to run snippets in the backend venv.

In [1]:
# --- repo-root anchor (identical in all four segment notebooks) ---
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing .git is found."""
    for d in [start, *start.parents]:
        if (d / ".git").exists():
            return d
    raise RuntimeError("Repo root (.git) not found above " + str(start))


REPO = find_repo_root(Path.cwd())
WARNERCO = REPO / "src" / "warnerco" / "backend"

print("REPO    :", REPO)
print("WARNERCO:", WARNERCO, "exists:", WARNERCO.exists())
assert WARNERCO.exists(), "WARNERCO backend not found - is REPO correct?"


def warnerco_py(code: str) -> str:
    """Run a Python snippet inside the WARNERCO backend venv + cwd.

    Raises if the snippet errors, so a broken probe fails loudly in validation
    instead of silently printing a traceback as if it were data.
    """
    r = subprocess.run(["uv", "run", "python", "-c", code], cwd=WARNERCO,
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("WARNERCO snippet failed:\n" + (r.stderr or r.stdout))
    noise = ("warning:", "warning", "deprecat", "pydanticdep", "onnxruntime",
             "tqdm", "huggingface", "resume_download", "class ", "dev-depe",
             "dependency-groups")
    return "\n".join(ln for ln in (r.stdout or "").splitlines()
                     if ln.strip() and not any(n in ln.lower() for n in noise))


print("Helper ready: warnerco_py(code)")

REPO    : C:\github\context-engineering
WARNERCO: C:\github\context-engineering\src\warnerco\backend exists: True
Helper ready: warnerco_py(code)


## Progressive tool loading in action

Instead of dumping 28 full schemas at the model (~9K tokens), `warn_search_tools` returns a tiny index, and `warn_describe_tool` returns one full schema on demand. This cell calls it live and shows the **default-limit gotcha**: a no-query call returns `count=20` (capped by the default `limit=20`), not the 26 non-meta tools or the full 28. Pass `limit=100` to see them all.

In [2]:
# Progressive loading: default-capped vs full inventory
print(warnerco_py(r'''
import asyncio
from app.mcp_tools import mcp
async def go():
    tools = await mcp.get_tools()
    default = await tools["warn_search_tools"].fn(query="", detail="name")
    print(f"no-query, default limit -> count={default.count} total={default.total}")
    full = await tools["warn_search_tools"].fn(query="", detail="name", limit=100)
    print(f"no-query, limit=100     -> count={full.count} total={full.total}")
    graph = await tools["warn_search_tools"].fn(query="graph", detail="summary")
    print(f"query=graph             -> count={graph.count} matched")
    for t in graph.tools[:4]:
        print("  " + t["name"] + ": " + t["summary"])
asyncio.run(go())
'''))

no-query, default limit -> count=20 total=28
no-query, limit=100     -> count=26 total=28
query=graph             -> count=8 matched
  warn_add_relationship: Add a relationship between two entities in the knowledge graph.
  warn_create_schematic: Create a new robot schematic in the database.
  warn_explain_schematic: Get an LLM-enriched explanation of a robot schematic.
  warn_graph_neighbors: Get all entities connected to the given entity in the knowledge graph.


## The same server, three clients

The cell below prints the real server entries from both checked-in config files so you can show the schema differences live. Key contrasts:

| | `.claude/mcp.json` (Claude Code) | `.vscode/mcp.json` (Copilot/VS Code) |
|---|---|---|
| Root key | `mcpServers` | `servers` |
| `type` field | optional | **required** (`stdio`/`http`) |
| Var expansion | `${VAR}` / `${VAR:-default}` | `${env:VAR}` / `${input:id}` |
| WARNERCO entries | 2 (general + pinned-CoALA) | 1 (`oreilly-warnerco-schematica`) + GitHub remote |

**Talk track:** "Same `uv run warnerco-mcp` binary. Claude Code registers it twice for teaching clarity; VS Code registers it once next to GitHub's remote MCP server. The IDE does not care whose server it is - it is all the same protocol."

In [3]:
# Read both client configs live and show the server keys
import json

claude_cfg = json.loads((REPO / ".claude" / "mcp.json").read_text(encoding="utf-8"))
vscode_cfg = json.loads((REPO / ".vscode" / "mcp.json").read_text(encoding="utf-8"))

print(".claude/mcp.json  root key 'mcpServers':", list(claude_cfg.get("mcpServers", {}).keys()))
print(".vscode/mcp.json  root key 'servers'   :", list(vscode_cfg.get("servers", {}).keys()))

.claude/mcp.json  root key 'mcpServers': ['warnerco-schematica-claude', 'warnerco-coala-memory']
.vscode/mcp.json  root key 'servers'   : ['oreilly-warnerco-schematica', 'github']
